# Nemotron-3 30B Fine-Tuning: Enhanced Reasoning Pipeline
## License Compliance

This project retains and complies with the Apache License 2.0 requirements of the original work.

- Original copyright and license notices are preserved.
- Modifications are documented in this file.
- No warranty is provided, as per the original license.
- 
> ### IMPORTANT
> **Attribution & Licensing**  
> This work is a derivative of the Kaggle notebook [“End-to-end finetuning for LB 0.85”](https://www.kaggle.com/code/huikang/end-to-end-finetuning-for-lb-0-85) by **Huikang**.  
> The original code is licensed under the **Apache License 2.0**. This modified version is provided in compliance with all original license terms.

---

## 🚀 Performance Comparison

By transitioning from standard investigation to a high-fidelity reasoning pipeline using the **nemotron-0-90** dataset, we achieved a significant jump in reasoning accuracy, particularly in hard categories like `cryptarithm` and `equation_guess`.

### **Summary of Gains**
| Metric | Original (Huikang) | Enhanced (taha) | Delta |
| :--- | :--- | :--- | :--- |
| **Total Accuracy** | 87.7% | **95.8%** | **+8.1%** |
| **Cryptarithm Deduce** | 8.2% | **89.8%** | **+81.6%** |
| **Cryptarithm Guess** | 6.7% | **85.4%** | **+78.7%** |
| **Equation Guess** | 15.4% | **92.6%** | **+77.2%** |

<details>
<summary>📊 View Detailed Metrics Comparison</summary>

#### Original Baseline
```text
================================================================
Category                      Found  Total   Accuracy     Avg ms
----------------------------------------------------------------
bit_manipulation               1364   1602      85.1%        0.8
cipher                         1576   1576     100.0%        0.1
cryptarithm_deduce               54    659       8.2%        0.0
cryptarithm_guess                11    164       6.7%        0.0
equation_numeric_deduce         540    596      90.6%        0.6
equation_numeric_guess           21    136      15.4%        0.7
gravity                        1597   1597     100.0%        0.1
numeral                        1576   1576     100.0%        0.0
unit_conversion                1594   1594     100.0%        0.1
----------------------------------------------------------------
TOTAL                          8333   9500      87.7%        0.2
================================================================
```

#### Enhanced Pipeline (Mine)
```text
================================================================
Category                      Found  Total   Accuracy     Avg ms
----------------------------------------------------------------
bit_manipulation               1364   1602      85.1%        0.8
cipher                         1576   1576     100.0%        0.1
cryptarithm_deduce              592    659      89.8%        0.0
cryptarithm_guess               140    164      85.4%        0.0
equation_numeric_deduce         540    596      90.6%        0.7
equation_numeric_guess          126    136      92.6%        0.7
gravity                        1597   1597     100.0%        0.1
numeral                        1576   1576     100.0%        0.0
unit_conversion                1594   1594     100.0%        0.1
----------------------------------------------------------------
TOTAL                          9105   9500      95.8%        0.2
================================================================
```
</details>

---

## 🛠️ Key Modifications

### 1. Dataset & Preprocessing
*   **CSV-Based Workflow**: Replaced the original corpus/token pipeline with a high-performance CSV-based dataset workflow.
*   **Prompt Engineering**: Introduced advanced prompt/response formatting using optimized chat templates.
*   **Dynamic Masking**: Implemented custom token masking logic that precisely targets assistant response boundaries for cleaner SFT.
*   **CoT Handling**: Robust Chain-of-Thought (CoT) integration using the **nemotron-0-90** regenerated dataset.

### 2. Environment & Dependency Handling
*   **Kaggle Optimization**: Added specific fixes for the Kaggle environment, including:
    *   Triton wheel installation and binary patching.
    *   **PTXAS** path fixes for cross-architecture compatibility.
    *   Automated management of CUDA-related binaries and shared libraries.

### 3. Training Pipeline Adjustments
*   **Streamlined Execution**: Simplified dataset loading and example construction to reduce VRAM overhead.
*   **LoRA Refinement**: Retained core LoRA parameters while cleaning up structural bottlenecks in the training loop.
*   **Modal Decoupling**: Removed unnecessary multi-environment branching to focus on a unified execution path.

### 4. Code Refactoring
*   **Simplified Logic**: Reduced complexity in environment-specific branching.
*   **Readability**: Extensive renaming and refactoring for a more maintainable codebase.

---

## 📝 Implementation Notes

### NOTE
> The core training strategy, LoRA integration, and optimization logic remain largely derived from the original implementation. This should be considered a **modified/derivative work**, not an independent reimplementation.

---

## 📂 Versioning

*   **Version 1**: Initial adaptation of the original baseline.
*   **Version 2**: Full runnable version with comprehensive environment fixes and verified results.
*   **Version 3**: Enhanced documentation and performance benchmarking (Current).

➡️ **Recommendation**: Please refer to **Version 2** for complete execution and final outputs.

In [ ]:
# -*- coding: utf-8 -*-
import os
import sys

# ── Shared config ─────────────────────────────────────────────────────
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

MAX_SEQ_LEN = 8192
NUM_STEPS = 1000
BATCH_SIZE = 32
MICRO_BATCH_SIZE = 4
LEARNING_RATE = 2e-4
RESET_WEIGHTS = True
IN_PROJ_ONLY = False
MOE_TIE_WEIGHTS = True
SHUFFLE_DATASET = False

MINUTES = 60

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "up_proj",
    "down_proj",
    "in_proj",
    "out_proj",
    "lm_head",
]

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    import os, glob, sys, subprocess, site, shutil, stat
    
    # ── Triton wheel setup ──────────────────────────────────────────────
    target = "/kaggle/working/pydeps"
    os.makedirs(target, exist_ok=True)
    
    triton_wheels = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
    if triton_wheels:
        subprocess.run(
            [
                sys.executable, "-m", "pip", "install",
                "--no-deps", "--target", target, "--upgrade", "--ignore-installed", triton_wheels[0]
            ],
            check=True
        )
    
    if target not in sys.path:
        sys.path.insert(0, target)
    site.addsitedir(target)
    
    # ── Environment fixes from 0.85 LB script ──────────────────────────
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
    
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    dst_bin = '/tmp/triton_nvidia_bin'
    
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        
        src_bin = os.path.dirname(ptxas_src)
        if not os.path.exists(dst_bin):
            shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
            for f in os.listdir(dst_bin):
                fp = os.path.join(dst_bin, f)
                if os.path.isfile(fp):
                    os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
                    
    # ALWAYS set these, even if files were already copied in a previous kernel run!
    os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
    
    try:
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
    except: pass
        
    try:
        import triton.backends.nvidia.compiler as nv_compiler
        nv_compiler.get_ptxas_version = lambda arch: '12.0'
    except: pass
    
    # ── Main Packages ──────────────────────────────────────────────────
    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    if os.path.isdir(packages_dir):
        # We append --ignore-installed to bypass the Errno 30 read-only file system issue you encountered earlier
        # while keeping the exact command structure of the 0.85 script.
        subprocess.run(
            [
                sys.executable, "-m", "pip", "install", "-q",
                "--no-index", "--find-links", packages_dir, "--ignore-installed",
                "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes"
            ],
            check=True,
        )
    
    # ── Mamba / Causal Conv1d ──────────────────────────────────────────
    def pick_last(wheels):
        return sorted(wheels)[-1] if wheels else None
    
    all_mamba = glob.glob("/kaggle/input/**/mamba_ssm-*.whl", recursive=True)
    all_causal = glob.glob("/kaggle/input/**/causal*conv1d*.whl", recursive=True)
    
    causal_wheel = pick_last(all_causal)
    mamba_wheel = pick_last(all_mamba)
    
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    
    # RTX wheels
    for _wd in ["/kaggle/input/datasets/llkh0a/rtx-wheels/wheels"]:
        if os.path.isdir(_wd):
            subprocess.run(
                [
                    sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", _wd, "--ignore-installed",
                    "protobuf==6.33.5", "sentencepiece", "safetensors", "huggingface_hub",
                ],
                check=False,
            )
    
    subprocess.run("rm -rf /kaggle/tmp/*", shell=True, check=True)

def run_training() -> None:
    import gc
    import json
    import math
    import random
    import time
    import pandas as pd
    import re
    from unsloth import FastLanguageModel
    import torch
    from cut_cross_entropy import linear_cross_entropy
    from peft import LoraConfig
    from peft.tuners.lora import Linear as LoraLinear

    if IS_KAGGLE:
        import kagglehub
        DATASET_PATH = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"
        if not os.path.exists(DATASET_PATH):
            DATASET_PATH = "/kaggle/input/nemotron-0-90/dataset_generated.csv"
        if not os.path.exists(DATASET_PATH):
            DATASET_PATH = "/kaggle/input/datasets/tahaalam2009/nemotron-0-90/problem_ids_matched.csv"
        MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
        OUTPUT_DIR = "."
    else:
        DATASET_PATH = "dataset_generated.csv"
        MODEL_PATH = "unsloth/Nemotron-3-Nano-30B-A3B"
        OUTPUT_DIR = "weights"

    cc = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}, sm_{cc[0] * 10 + cc[1]}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    print("Loading base model and tokenizer...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=True,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    
    print(f"Loading dataset from {DATASET_PATH}...")
    df = pd.read_csv(DATASET_PATH)
    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
    
    examples = []
    
    for _, row in df.iterrows():
        prompt = str(row["prompt"])
        answer = str(row["answer"])
        cot = str(row["generated_cot"])
        if not cot or cot == "nan" or len(cot.strip()) < 5:
            continue
            
        cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
        user_content = prompt + PROMPT_SUFFIX
        assistant_content = "<think>\n" + cot_cleaned + f"\n</think>\n\\boxed{{{answer}}}"
        
        full_str = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": assistant_content}
            ],
            tokenize=False,
            add_generation_prompt=False
        )
        
        encoded = tokenizer(full_str, return_offsets_mapping=True, add_special_tokens=False)
        input_ids = encoded["input_ids"]
        offsets = encoded["offset_mapping"]
        
        target_substr = "<|im_start|>assistant\n"
        substr_idx = full_str.find(target_substr)
        if substr_idx == -1:
            print(f"Warning: could not find assistant start in row {row.get('id', 'unknown')}")
            continue
            
        prompt_end_char = substr_idx + len(target_substr)
        
        mask_end_token_idx = -1
        for i, (start, end) in enumerate(offsets):
            if start >= prompt_end_char:
                mask_end_token_idx = i
                break
                
        if mask_end_token_idx == -1:
            mask_end_token_idx = len(input_ids)
            
        tokens = input_ids
        mask = [0.0] * mask_end_token_idx + [1.0] * (len(tokens) - mask_end_token_idx)
        
        if len(tokens) > MAX_SEQ_LEN:
            tokens = tokens[:MAX_SEQ_LEN]
            mask = mask[:MAX_SEQ_LEN]
            
        if sum(mask) == 0:
            continue
            
        examples.append({
            "problem_id": str(row["id"]) if "id" in row else str(len(examples)),
            "tokens": tokens[:-1],
            "targets": tokens[1:],
            "weights": mask[1:],
        })
        
    total_unmasked = sum(sum(e["weights"]) for e in examples)
    total_tokens = sum(len(e["tokens"]) for e in examples)
    print(f"Loaded {len(examples)} examples, {total_tokens:,} tokens (unmasked={total_unmasked:,.0f})")

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    FastLanguageModel.for_training(model)

    nemotron_mod = None
    for _name, _m in sys.modules.items():
        if "modeling_nemotron_h" in _name and hasattr(_m, "is_fast_path_available"):
            nemotron_mod = _m
            break
    if nemotron_mod is not None:
        nemotron_mod.is_fast_path_available = True
        print("Patched is_fast_path_available = True")

    _causal_lm = model
    while hasattr(_causal_lm, "model"):
        _causal_lm = _causal_lm.model
    _lm_head = _causal_lm.lm_head
    if not isinstance(_lm_head, LoraLinear):
        _cfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
        model.base_model._create_and_replace(_cfg, "default", target=_lm_head, target_name="lm_head", parent=_causal_lm)
        print("Manually added LoRA to lm_head")

    for name, param in model.named_parameters():
        if ".lora_" in name:
            param.data = param.data.to(torch.float32)

    for name, param in model.named_parameters():
        if ".lora_" in name:
            continue
        is_router = ".mixer.gate." in name
        if is_router:
            continue
        assert param.dtype == torch.bfloat16, f"param {name} expected bf16, got {param.dtype}"
        
    _base = model
    while hasattr(_base, "model"):
        _base = _base.model

    def _patched_causal_forward(input_ids=None, attention_mask=None, labels=None, **kwargs):
        backbone_out = _base.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **{k: v for k, v in kwargs.items() if k in ("position_ids", "past_key_values", "use_cache")}
        )
        hidden_states = backbone_out[0]
        lm_head = _base.lm_head
        base_w = lm_head.base_layer.weight
        lora_A = lm_head.lora_A["default"].weight
        lora_B = lm_head.lora_B["default"].weight
        scaling = lm_head.scaling["default"]
        lm_weight = base_w + scaling * lora_B @ lora_A
        if labels is not None:
            per_token_ce = linear_cross_entropy(hidden_states, lm_weight, labels, reduction="none")
            loss = per_token_ce.mean()
        else:
            per_token_ce = None
            loss = None
        model._cached_per_token_ce = per_token_ce
        return loss

    _base.forward = _patched_causal_forward
    print("Patched CausalLM.forward with CCE")

    moe_tied_params = []
    if MOE_TIE_WEIGHTS:
        w1_proj_names = ("gate_up_proj", "up_proj", "gate_proj", ".w1.")
        w2_proj_names = ("down_proj", ".w2.")
        for name, param in model.named_parameters():
            if not param.requires_grad: continue
            if ".experts." not in name or ".lora_" not in name: continue
            is_w1 = any(p in name for p in w1_proj_names)
            is_w2 = any(p in name for p in w2_proj_names)
            is_A = ".lora_A." in name
            is_B = ".lora_B." in name
            should_tie = (is_w1 and is_A) or (is_w2 and is_B)
            if not should_tie: continue
            if param.dim() < 2 or param.shape[0] <= 1: continue
            moe_tied_params.append(param)

        def _tie_param_init():
            with torch.no_grad():
                for p in moe_tied_params:
                    mean = p.data.mean(dim=0, keepdim=True)
                    p.data.copy_(mean.expand_as(p.data))

        def _tie_grads():
            with torch.no_grad():
                for p in moe_tied_params:
                    if p.grad is None: continue
                    grad_sum = p.grad.sum(dim=0, keepdim=True)
                    p.grad.copy_(grad_sum.expand_as(p.grad))

        _tie_param_init()
    else:
        def _tie_grads(): pass

    gc.collect()
    torch.cuda.empty_cache()

    device = next(model.parameters()).device
    optimizer = None

    indices = list(range(len(examples)))
    if SHUFFLE_DATASET:
        rng = random.Random(0)
        rng.shuffle(indices)

    max_steps = len(examples) // BATCH_SIZE
    num_steps = min(NUM_STEPS, max_steps)

    step = 0
    for batch_start in range(0, len(indices), BATCH_SIZE):
        if step >= num_steps:
            break
        batch_indices = indices[batch_start : batch_start + BATCH_SIZE]
        batch = [examples[i] for i in batch_indices]
        batch_tokens = [e["tokens"] for e in batch]
        batch_targets = [e["targets"] for e in batch]
        batch_weights = [e["weights"] for e in batch]

        n = len(batch)
        n_accum = math.ceil(n / MICRO_BATCH_SIZE)
        total_loss_sum = 0.0
        total_weight_sum = 0.0

        for mb_start in range(0, n, MICRO_BATCH_SIZE):
            mb_end = min(mb_start + MICRO_BATCH_SIZE, n)
            mb_toks = batch_tokens[mb_start:mb_end]
            mb_tgts = batch_targets[mb_start:mb_end]
            mb_wts = batch_weights[mb_start:mb_end]

            n_micro = len(mb_toks)
            max_len = max(len(t) for t in mb_toks)
            total_len = sum(len(t) for t in mb_toks)

            padded_input = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)
            padded_targets = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)
            padded_weights = torch.zeros(n_micro, max_len, dtype=torch.float32, device=device)
            attention_mask = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)
            
            for i in range(n_micro):
                seq_len = len(mb_toks[i])
                padded_input[i, :seq_len] = torch.tensor(mb_toks[i], dtype=torch.long)
                padded_targets[i, :seq_len] = torch.tensor(mb_tgts[i], dtype=torch.long)
                padded_weights[i, :seq_len] = torch.tensor(mb_wts[i], dtype=torch.float32)
                attention_mask[i, :seq_len] = 1

            t0 = time.time()
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                model(
                    input_ids=padded_input,
                    attention_mask=attention_mask,
                    labels=padded_targets,
                    use_cache=False,
                )
                per_token_ce = model._cached_per_token_ce
                weighted_loss = per_token_ce * padded_weights
                weight_sum_t = padded_weights.sum()
                loss_sum_t = weighted_loss.sum()
                loss = loss_sum_t / weight_sum_t if weight_sum_t > 0 else loss_sum_t * 0.0

            (loss / n_accum).backward()
            total_loss_sum += loss_sum_t.item()
            total_weight_sum += weight_sum_t.item()
            del loss, per_token_ce, weighted_loss

            t_end = time.time()
            peak_gb = torch.cuda.max_memory_allocated() / 1e9
            mem_gb = torch.cuda.memory_allocated() / 1e9
            mb_idx = mb_start // MICRO_BATCH_SIZE
            print(
                f"    micro-batch {mb_idx}: {n_micro} seqs, max_len={max_len}, "
                f"total_len={total_len}, wall={t_end - t0:.1f}s, "
                f"peak={peak_gb:.1f}GB, mem={mem_gb:.1f}GB"
            )

        if optimizer is None:
            optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=LEARNING_RATE,
                betas=(0.9, 0.95),
                eps=1e-8,
                weight_decay=0.0,
            )
        lr = LEARNING_RATE * (1 - step / num_steps)
        for pg in optimizer.param_groups:
            pg["lr"] = lr
        _tie_grads()
        grad_norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], max_norm=1e9)
        optimizer.step()
        optimizer.zero_grad()
        loss_mean = total_loss_sum / total_weight_sum if total_weight_sum > 0 else 0
        step += 1
        print(f"  step {step}/{num_steps}: loss:mean={loss_mean:.6f}, grad_norm={grad_norm:.4f}, lr={lr:.2e}")

    print(f"\nTraining complete. Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

    from safetensors.torch import load_file, save_file
    save_dir = "." if IS_KAGGLE else OUTPUT_DIR
    os.makedirs(save_dir, exist_ok=True)
    for _f in os.listdir(save_dir):
        if _f.startswith("adapter"):
            os.remove(os.path.join(save_dir, _f))
    model.save_pretrained(save_dir)
    st_path = os.path.join(save_dir, "adapter_model.safetensors")
    tensors = load_file(st_path)
    renamed = {k.replace("base_model.model.lm_head.", "base_model.model.backbone.lm_head."): v for k, v in tensors.items()}
    save_file(renamed, st_path)

    if IS_KAGGLE:
        import zipfile
        adapter_files = [f for f in os.listdir(save_dir) if f.startswith("adapter")]
        SUBMISSION_ZIP = "submission.zip"
        with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
            for fname in adapter_files:
                zf.write(os.path.join(save_dir, fname), fname)
        for fname in adapter_files:
            os.remove(os.path.join(save_dir, fname))
        print(f"Wrote {SUBMISSION_ZIP}")

    print("Training complete.")

if __name__ == "__main__":
    run_training()
